# Agent: Control Agent (Decision Logic)

**Purpose:** Monitor trigger events and water level readings; decide on evacuation alerts.

**Decision Logic:**
- If water_level ≥ 1.0m → Issue HIGH alert (evacuation)
- If water_level < 1.0m → Issue LOW alert (all-clear)

**Subscribes to:** `city/flood/trigger` topic

**Publishes to:** `city/flood/control/command` topic as JSON `ControlCommand` messages

In [1]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import TriggerEvent, ControlCommand

# Load configuration
try:
    cfg = load_config()
    print(f"MQTT Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Base Topic: {cfg.mqtt.base_topic}")
except Exception as e:
    print(f"Error loading config: {e}")
    cfg = None

# Control parameters
WATER_THRESHOLD = 1.0  # Alert threshold in meters
BASELINE_WATER = 0.2   # Normal water level
FLOOD_WATER = 6.5      # Maximum water during flood

# State tracking
current_trigger = None
current_water_level = BASELINE_WATER
current_alert_status = False
update_count = 0

print(f"\nControl agent configured:")
print(f"  Water threshold: {WATER_THRESHOLD}m")
print(f"  Flood level: {FLOOD_WATER}m")
print(f"  Baseline level: {BASELINE_WATER}m")

MQTT Broker: 127.0.0.1:1883
Base Topic: simulated-city

Control agent configured:
  Water threshold: 1.0m
  Flood level: 6.5m
  Baseline level: 0.2m


In [7]:
if cfg:
    mqtt = MqttConnector(cfg.mqtt, client_id_suffix='control')
    mqtt.connect()
    if mqtt.wait_for_connection(timeout=5):
        print("✓ Connected to MQTT broker")
    else:
        print("✗ Failed to connect to MQTT broker")
        mqtt = None


def on_trigger_message(client, userdata, msg):
    """Callback when trigger event is received."""
    global current_trigger
    try:
        payload = msg.payload.decode()
        data = json.loads(payload)
        current_trigger = TriggerEvent.from_json_dict(data)
        print(f"Trigger received: {current_trigger.severity} {current_trigger.event}")
    except Exception as e:
        print(f"Error parsing trigger: {e}")


# Subscribe to trigger events
if mqtt:
    mqtt.client.subscribe(make_topic(cfg.mqtt, "trigger"), qos=1)
    mqtt.client.message_callback_add(
        make_topic(cfg.mqtt, "trigger"),
        on_trigger_message
    )
    print("Subscribed to: city/flood/trigger")

✓ Connected to MQTT broker
Subscribed to: city/flood/trigger


Error parsing trigger: type object 'TriggerEvent' has no attribute 'from_json_dict'


In [8]:
def publish_control_command(action: str, target: str = "all"):
    """Publish a control command to Response agent."""
    if not mqtt:
        return False
    
    cmd = ControlCommand(
        action=action,
        target=target,
        parameters={"threshold": WATER_THRESHOLD},
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(cfg.mqtt, "control", "command")
    payload = cmd.to_json()
    
    mqtt.client.publish(topic, payload, qos=1)
    return True


def update_water_level():
    """Simulate water level based on current trigger."""
    global current_water_level
    
    if not current_trigger:
        # No trigger: water stays at baseline
        current_water_level = BASELINE_WATER
    elif current_trigger.severity == "low":
        # Low severity: water drops back to baseline quickly
        current_water_level = max(current_water_level - 0.5, BASELINE_WATER)
    elif current_trigger.severity == "high":
        # High severity: water rises toward flood level
        # Ramp up to reach 6.5m within 15 seconds (step by 0.5 per second)
        current_water_level = min(current_water_level + 0.5, FLOOD_WATER)
    
    return current_water_level


def check_alert_status() -> tuple[bool, str]:
    """Determine if alert should be active based on water level."""
    if current_water_level >= WATER_THRESHOLD:
        return True, "alert"
    else:
        return False, "alert"


def update_control_state():
    """Update control state and issue commands if needed."""
    global current_alert_status, update_count
    
    # Update water level
    water_level = update_water_level()
    
    # Check if alert status should change
    new_alert_status, action = check_alert_status()
    
    # Print status
    trigger_str = f"{current_trigger.severity}" if current_trigger else "None"
    update_count += 1
    alert_indicator = "✓" if new_alert_status else "✗"
    print(f"[{update_count}] Water: {water_level:.2f}m | Trigger: {trigger_str} | Alert: {new_alert_status}")
    
    # If alert status changed, publish command
    if new_alert_status != current_alert_status:
        if new_alert_status:
            print(f"  ⚠️  ALERT: Water level {water_level:.2f}m >= threshold {WATER_THRESHOLD}m")
            print(f"    → Published control command: {action} (high)")
            publish_control_command("alert", "evacuation")
        else:
            print(f"  ✓ ALL-CLEAR: Water level {water_level:.2f}m < threshold")
            print(f"    → Published control command: {action} (low)")
            publish_control_command("alert", "return")
        current_alert_status = new_alert_status

In [9]:
print("\nStarting control loop...")
print("Update interval: 1.0s")
print("Press Ctrl+C to stop\n")

try:
    while True:
        update_control_state()
        time.sleep(1)
        
except KeyboardInterrupt:
    print("\n\n✓ Control agent stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()


Starting control loop...
Update interval: 1.0s
Press Ctrl+C to stop

[1] Water: 0.20m | Trigger: None | Alert: False
[2] Water: 0.20m | Trigger: None | Alert: False
[3] Water: 0.20m | Trigger: None | Alert: False
[4] Water: 0.20m | Trigger: None | Alert: False
[5] Water: 0.20m | Trigger: None | Alert: False
[6] Water: 0.20m | Trigger: None | Alert: False
[7] Water: 0.20m | Trigger: None | Alert: False
[8] Water: 0.20m | Trigger: None | Alert: False
[9] Water: 0.20m | Trigger: None | Alert: False
[10] Water: 0.20m | Trigger: None | Alert: False
[11] Water: 0.20m | Trigger: None | Alert: False
[12] Water: 0.20m | Trigger: None | Alert: False
[13] Water: 0.20m | Trigger: None | Alert: False
Error parsing trigger: type object 'TriggerEvent' has no attribute 'from_json_dict'
[14] Water: 0.20m | Trigger: None | Alert: False
[15] Water: 0.20m | Trigger: None | Alert: False
[16] Water: 0.20m | Trigger: None | Alert: False
[17] Water: 0.20m | Trigger: None | Alert: False
[18] Water: 0.20m | Tr

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


In [18]:
UPDATE_INTERVAL = 1.0  # Update decision every 1 second
RAMP_RATE_UP = FLOOD_WATER_LEVEL / RAMP_UP_TIME  # meters per second
RAMP_RATE_DOWN = (FLOOD_WATER_LEVEL - BASELINE_WATER_LEVEL) / RAMP_DOWN_TIME  # meters per second

print("Starting control loop...")
print(f"Update interval: {UPDATE_INTERVAL}s\n")

loop_count = 0
last_update_time = time.time()

try:
    while True:
        loop_count += 1
        current_time = time.time()
        elapsed = current_time - last_update_time
        
        with state["lock"]:
            severity = state["last_trigger_severity"]
            current_level = state["current_water_level"]
        
        # Update water level based on trigger severity
        if severity == "high":
            # Ramp up to flood level
            target = FLOOD_WATER_LEVEL
            desired_level = min(current_level + RAMP_RATE_UP * elapsed, target)
        elif severity == "low":
            # Stay at baseline
            desired_level = BASELINE_WATER_LEVEL
        else:
            # Recover gradually
            desired_level = max(current_level - RAMP_RATE_DOWN * elapsed, BASELINE_WATER_LEVEL)
        
        with state["lock"]:
            state["current_water_level"] = desired_level
        
        # Check if we crossed threshold
        should_alert = desired_level >= WATER_LEVEL_THRESHOLD
        was_alerted = state["alerted"]
        
        if should_alert and not was_alerted:
            print(f"[{loop_count}] ⚠️  ALERT: Water level {desired_level:.2f}m >= threshold {WATER_LEVEL_THRESHOLD}m")
            publish_alert_command("alert", "high")
            state["alerted"] = True
        elif not should_alert and was_alerted:
            print(f"[{loop_count}] ✓ ALL-CLEAR: Water level {desired_level:.2f}m < threshold")
            publish_alert_command("alert", "low")
            state["alerted"] = False
        else:
            print(f"[{loop_count}] Water: {desired_level:.2f}m | Trigger: {severity} | Alert: {state['alerted']}")
        
        last_update_time = current_time
        time.sleep(UPDATE_INTERVAL)

except KeyboardInterrupt:
    print("\n\nControl loop stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

Starting control loop...
Update interval: 1.0s

[1] Water: 0.20m | Trigger: None | Alert: False
[2] Water: 0.20m | Trigger: None | Alert: False
[3] Water: 0.20m | Trigger: None | Alert: False
[4] Water: 0.20m | Trigger: None | Alert: False
[5] Water: 0.20m | Trigger: None | Alert: False
[6] Water: 0.20m | Trigger: None | Alert: False
[7] Water: 0.20m | Trigger: None | Alert: False
[8] Water: 0.20m | Trigger: None | Alert: False
[9] Water: 0.20m | Trigger: None | Alert: False
[10] Water: 0.20m | Trigger: None | Alert: False
[11] Water: 0.20m | Trigger: None | Alert: False
[12] Water: 0.20m | Trigger: None | Alert: False
[13] Water: 0.20m | Trigger: None | Alert: False
[14] Water: 0.20m | Trigger: None | Alert: False
[15] Water: 0.20m | Trigger: None | Alert: False
[16] Water: 0.20m | Trigger: None | Alert: False
[17] Water: 0.20m | Trigger: None | Alert: False
[18] Water: 0.20m | Trigger: None | Alert: False
[19] Water: 0.20m | Trigger: None | Alert: False
[20] Water: 0.20m | Trigger: N

## Control Loop

Main decision loop that:
1. Checks trigger severity
2. Updates water level (ramp up/down)
3. Issues alerts/all-clear commands based on threshold

In [19]:
def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_alert_command(action, severity):
    """Publish a control command to response agents."""
    cmd = ControlCommand(
        action=action,
        target="all_pedestrians",
        parameters={"severity": severity},
        timestamp=get_iso_timestamp()
    )
    topic = make_topic(mqtt_cfg, "control", "command")
    payload = json.dumps(cmd.to_json())
    publisher.publish_json(topic, payload, qos=1)
    print(f"  → Published control command: {action} ({severity})")
    return cmd

def on_trigger_message(client, userdata, msg):
    """Callback when trigger event received."""
    try:
        data = json.loads(msg.payload.decode())
        evt = TriggerEvent.from_json(data)
        
        with state["lock"]:
            state["last_trigger_severity"] = evt.severity
        
        print(f"[{evt.timestamp}] Trigger received: {evt.severity.upper()} {evt.event}")
    except Exception as e:
        print(f"Error parsing trigger: {e}")

# Register MQTT callbacks
connector.client.on_message = on_trigger_message
trigger_topic = make_topic(mqtt_cfg, "trigger")
connector.client.subscribe(trigger_topic)

print(f"Subscribed to: {trigger_topic}")

Subscribed to: simulated-city/trigger


## Decision Logic and Callbacks

This section defines the logic for:
1. Receiving trigger events
2. Updating water level simulation
3. Issuing alert commands to response agents

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="control")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

✓ Connected to MQTT broker


## Connect to MQTT Broker

In [2]:
import time
import json
import threading
from datetime import datetime, timezone

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent, ControlCommand

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

# Control thresholds
WATER_LEVEL_THRESHOLD = 1.0  # meters - trigger evacuation
FLOOD_WATER_LEVEL = 6.5  # meters during high trigger
BASELINE_WATER_LEVEL = 0.2  # meters baseline
RAMP_UP_TIME = 5.0  # seconds to reach flood level
RAMP_DOWN_TIME = 10.0  # seconds to recover

# State tracking
state = {
    "last_trigger_severity": None,
    "current_water_level": BASELINE_WATER_LEVEL,
    "alerted": False,
    "lock": threading.Lock()
}

print("Control agent configured:")
print(f"  Water threshold: {WATER_LEVEL_THRESHOLD}m")
print(f"  Flood level: {FLOOD_WATER_LEVEL}m")
print(f"  Baseline level: {BASELINE_WATER_LEVEL}m")

Control agent configured:
  Water threshold: 1.0m
  Flood level: 6.5m
  Baseline level: 0.2m


## Setup and Configuration

# Agent: Control (Decision Logic)

**Role:** Central decision engine for the flood response system.

**Responsibilities:**
1. Subscribe to trigger events and sensor data
2. Simulate water level changes based on trigger severity
3. Monitor water level threshold (1m)
4. Issue evacuation commands when threshold exceeded
5. Coordinate with response agents

**Decision Logic:**
- LOW trigger → Water stays baseline
- HIGH trigger → Water rises to 6-7m
- Water >= 1m → Issue "alert" command with severity
- Water recovers → Issue "all-clear" command

# Agent Control
Logic notebook that subscribes to observer data and trigger events, then emits `ControlCommand` messages.